# CN7 가스: 정상 대비 동시 변화 보조 전문가
기존 지도 앙상블과 정상분포 통계 전문가의 결합. 평가 최적 후보는 개발 성적이다.


In [1]:
from pathlib import Path
import nbformat
old=nbformat.read("notebooks/170_cn7_gas_persistent_relation_experts.ipynb",as_version=4)
oldcode="\n".join(c.source for c in old.cells if c.cell_type=="code")
exec(oldcode[:oldcode.index("def predict(parent,q):")])
exec(oldcode[oldcode.index("def stats(a,b):"):oldcode.index("records=[];choices=[]")])
BASIC=["Mold_Temperature_3","Max_Injection_Speed","Filling_Time","Max_Injection_Pressure",
 "Plasticizing_Position","Plasticizing_Time","Cushion_Position","Average_Back_Pressure","Max_Screw_RPM"]
raw=d[[v+"_raw" for v in BASIC]].to_numpy(float)
def features(ctx):
 ii=np.flatnonzero(ctx);a=pd.DataFrame(raw[ii],index=ii)
 segment=np.cumsum(np.r_[True,(g[ii][1:]!=g[ii][:-1])|(np.diff(ii)!=1)])
 med=a.groupby(segment,group_keys=False).apply(lambda b:b.shift(1).rolling(10,min_periods=3).median())
 delta=a-med
 # Only past sensor values, never past quality labels.
 return pd.DataFrame({"cold_now":-a[0],"cooling":-delta[0],"speed":a[1],"short_fill":-a[2],
  "low_pressure":-a[3],"meter":a[4],"plasticizing":a[5],"cushion":a[6],
  "backpressure":a[7],"low_rpm":-a[8],"speed_rise":delta[1],"fill_drop":-delta[2],
  "meter_rise":delta[4],"cushion_rise":delta[6]})
def ranked_scores(ctx,qctx,ref,q):
 tr=features(ctx).reindex(ref)
 te=features(qctx).reindex(q)
 ranks=np.zeros((len(q),len(tr.columns)))
 for run in pd.unique(g[q]):
  ni=np.flatnonzero((y[ref]==0)&(g[ref]==run))
  if len(ni)<8:ni=np.flatnonzero(y[ref]==0)
  pos=np.flatnonzero(g[q]==run)
  for k,c in enumerate(tr):
   values=np.sort(tr[c].iloc[ni].dropna().to_numpy(float))
   if not len(values):ranks[pos,k]=.5;continue
   v=te[c].iloc[pos].to_numpy(float)
   left=np.searchsorted(values,v,side="left");right=np.searchsorted(values,v,side="right")
   ranks[pos,k]=(left+right)/(2*len(values))
   ranks[pos[np.isnan(v)],k]=.5
 r={c:ranks[:,k] for k,c in enumerate(tr.columns)}
 pair=lambda a,b:np.minimum(r[a],r[b])
 return {
 "cold_fast":pair("cold_now","speed"),
 "cooling_fast":pair("cooling","speed"),
 "cooling_shortfill":pair("cooling","short_fill"),
 "speed_shortfill":pair("speed","short_fill"),
 "pressure_fast":pair("low_pressure","speed"),
 "meter_longplastic":pair("meter","plasticizing"),
 "cushion_longplastic":pair("cushion","plasticizing"),
 "load_highbp_lowrpm":pair("backpressure","low_rpm"),
 "transient_fast_fill":pair("speed_rise","fill_drop"),
 "cold_flow_max":np.maximum(pair("cooling","speed"),pair("cooling","short_fill"))}
memo={};audit=[]
def predict(parent,q):
 key=(tuple(np.flatnonzero(parent)),tuple(q))
 if key in memo:return memo[key]
 ctx=train_context(parent,q);ref=np.flatnonzero(ctx&eligible)
 assert not ctx[q].any() and set(ref).isdisjoint(q)
 audit.append({"timestamp_overlap":len(set(d.TimeStamp.iloc[ref])&set(d.TimeStamp.iloc[q])),
 "test_raw_in_training_history":int(ctx[q].sum()),"train_gas":int(y[ref].sum())})
 ans=ranked_scores(ctx,parent,ref,q)
 memo[key]=(ans,ref,ctx)
 return memo[key]
b=pd.read_csv(SOURCE/"cn7_165_gas_supervised_complementary_predictions.csv",parse_dates=["TimeStamp"]).set_index("TimeStamp").loc[d.TimeStamp]
base165=b.three_specialists_confirmed.to_numpy(int)
a=pd.read_csv(OUT/"cn7_170_persistence_predictions.csv",parse_dates=["TimeStamp"])
a=a[(a.model=="persistent_abs4_ET")&(a.rule=="normal95")].set_index("TimeStamp").loc[d.TimeStamp]
base170=base165|a.prediction.to_numpy(int)
a=pd.read_csv(OUT/"cn7_171_flow_predictions.csv",parse_dates=["TimeStamp"])
a=a[(a.model=="flow_signed6_LR")&(a.rule=="normal99")].set_index("TimeStamp").loc[d.TimeStamp]
base171=base170|a.prediction.to_numpy(int)
records=[];choices=[]
for held in pd.unique(blocks):
 q=np.flatnonzero(blocks==held);outer,ref,ctx=predict(np.ones(N,bool),q)
 inner={n:np.full(len(ref),np.nan) for n in outer}
 for ib in pd.unique(innerblocks[ref]):
  mask=innerblocks[ref]==ib;ip,_,_=predict(ctx,ref[mask])
  for n in inner:inner[n][mask]=ip[n]
 options=[]
 for name in outer:
  for quant in [.95,.97,.99,.995]:
   cut=float(np.nextafter(np.quantile(inner[name][y[ref]==0],quant,method="higher"),np.inf))
   met=stats(y[ref],inner[name]>=cut)
   options.append((met["F1"],met["TP"],-met["FP"],name,quant,cut))
   for j,i in enumerate(q):records.append({"model":name,"rule":f"inner_normal{quant}",
    "TimeStamp":d.TimeStamp.iloc[i],"actual":int(y[i]),"eligible":int(eligible[i]),
    "score":float(outer[name][j]),"threshold":cut,"prediction":int(outer[name][j]>=cut)})
 best=max(options);_,_,_,name,quant,cut=best
 choices.append({"heldout_block":held,"model":name,"quantile":quant,"inner_F1":best[0]})
 for j,i in enumerate(q):records.append({"model":"inner_selected","rule":"inner_selection",
  "TimeStamp":d.TimeStamp.iloc[i],"actual":int(y[i]),"eligible":int(eligible[i]),
  "score":float(outer[name][j]),"threshold":cut,"prediction":int(outer[name][j]>=cut)})
 print(f"Finished normal joint {len(choices)}/20",flush=True)
pp=pd.DataFrame(records);rows=[]
for (name,rule),a in pp.groupby(["model","rule"],sort=False):
 a=a.set_index("TimeStamp").loc[d.TimeStamp];pr=a.prediction.to_numpy(int)
 for policy,base in [("standalone",np.zeros(N,int)),("add_to_165",base165),("add_to_170",base170),("add_to_171",base171)]:
  pred=pr|base
  rows.append({"model":name,"rule":rule,"policy":policy,"N":605,"gas":13,**stats(y[eligible],pred[eligible])})
mm=pd.DataFrame(rows)
pp.to_csv(OUT/"cn7_172_normal_joint_predictions.csv",index=False,encoding="utf-8-sig")
mm.to_csv(OUT/"cn7_172_normal_joint_metrics.csv",index=False,encoding="utf-8-sig")
pd.DataFrame(audit).to_csv(OUT/"cn7_172_normal_joint_audit.csv",index=False)
pd.DataFrame(choices).to_csv(OUT/"cn7_172_normal_joint_inner_choices.csv",index=False)
assert pd.DataFrame(audit).timestamp_overlap.eq(0).all()
assert pd.DataFrame(audit).test_raw_in_training_history.eq(0).all()
top=mm.sort_values(["F1","FP"],ascending=[False,True]).head(15)
best=top.iloc[0]
a=pp[(pp.model==best.model)&(pp.rule==best.rule)].set_index("TimeStamp").loc[d.TimeStamp]
base={"standalone":np.zeros(N,int),"add_to_165":base165,"add_to_170":base170,"add_to_171":base171}[best.policy]
pr=a.prediction.to_numpy(int)|base
cases=d[["TimeStamp","gas","any_fail","run_id","seq"]].copy()
cases["baseline"]=base;cases["candidate"]=pr
cases["score"]=a.score.to_numpy();cases["threshold"]=a.threshold.to_numpy()
cases.to_csv(OUT/"cn7_172_normal_joint_best_cases.csv",index=False,encoding="utf-8-sig")
print("\nTOP\n",top.to_string(index=False))
print("\nINNER SELECTED\n",mm[mm.model.eq("inner_selected")].to_string(index=False))
print("\nGAS\n",cases[cases.gas].to_string(index=False))
print("\nNEW ALARMS\n",cases[cases.candidate.ne(cases.baseline)].to_string(index=False))
print("\nFULL COHORT",stats(y,pr))
lines=["# CN7 가스: 정상 기반 동시 변화 보조 전문가","",
 "지도학습 모델을 고정하고 정상 분포 기반 2변수 tail coincidence 전문가를 추가했다.",
 "각 센서의 학습 정상 대비 백분위를 계산하고 두 신호 백분위의 최소값으로 동시성을 측정한다.",
 "냉각+속도, 냉각+짧은 충전, 계량/쿠션+긴 가소화, 배압+RPM 등을 검토했다.",
 "과거 변화는 현재 샷을 제외한 과거10샷 중앙값 대비. 평가 기록은 학습 이력에서 제거.",
 "정상 순위 모델은 비지도/통계 보조 모델이며 전체를 순수 지도학습이라고 부르면 안 된다.",
 "같은20 시간블록/purge10/nested5. 미래 시점이 학습에 포함될 수 있다.",
 "고정 최고는 외부 평가 결과로 선택한 개발 성적이며 최종 독립 성능이 아니다.","",
 "## 고정 후보",top.to_string(index=False),"",
 "## 안쪽 선택",mm[mm.model.eq("inner_selected")].to_string(index=False),"",
 "## 가스13개",cases[cases.gas].to_string(index=False),"",
 "## 새 경보",cases[cases.candidate.ne(cases.baseline)].to_string(index=False),"",
 "## 순수 미성형까지 포함한 전체606샷 가스 판정",str(stats(y,pr))]
(ROOT/"reports/cn7_172_normal_joint_review.md").write_text("\n".join(lines),encoding="utf-8")
(OUT/"cn7_172_normal_joint_provenance.json").write_text(json.dumps({
 "input_sha256":hashlib.sha256(INPUT.read_bytes()).hexdigest(),"validation":"same20/purge10/nested5",
 "cutoffs":"inner normal95/97/99/99.5 quantiles; no evaluation-label threshold fit",
 "caveat":"test-informed development features and post-hoc candidate selection",
 "heads":"normal empirical ranks + fixed physical pair minimum; supervised base165/170/171"},ensure_ascii=False,indent=2),encoding="utf-8")


Finished normal joint 1/20


Finished normal joint 2/20


Finished normal joint 3/20


Finished normal joint 4/20


Finished normal joint 5/20


...

Finished normal joint 18/20


Finished normal joint 19/20


Finished normal joint 20/20



TOP
               model              rule     policy   N  gas  TP  FP  FN  TN  precision   recall       F1
          cold_fast inner_normal0.995 add_to_171 605   13  10  11   3 581   0.476190 0.769231 0.588235
          cold_fast inner_normal0.995 add_to_170 605   13   8   7   5 585   0.533333 0.615385 0.571429
transient_fast_fill inner_normal0.995 add_to_171 605   13   9  10   4 582   0.473684 0.692308 0.562500
       cooling_fast inner_normal0.995 add_to_171 605   13  10  13   3 579   0.434783 0.769231 0.555556
      cold_flow_max inner_normal0.995 add_to_171 605   13  10  13   3 579   0.434783 0.769231 0.555556
  cooling_shortfill inner_normal0.995 add_to_171 605   13   9  11   4 581   0.450000 0.692308 0.545455
    speed_shortfill inner_normal0.995 add_to_171 605   13   9  11   4 581   0.450000 0.692308 0.545455
  meter_longplastic inner_normal0.995 add_to_171 605   13   9  11   4 581   0.450000 0.692308 0.545455
 load_highbp_lowrpm inner_normal0.995 add_to_171 605   13   9  11  

395